# 02.05 Encoding and Numerical Preprocessing
**Module:** Feature Engineering & Preprocessing Pipeline  
**Target Dataset:** `data/interim/cleaned/ofphil_cleaned_tabular.csv`  
**Expected Splits:** `data/processed/splits/{train, validation, test}/`  
**Expected Pipeline:** `data/processed/features/preprocessing_pipeline.joblib`  
**Expected Audit Artifact:** `data/processed/preprocessing/numerical_scaling_stats.csv`

---
### Research Objectives
1. **Feature Subset Selection & Protected Attribute Isolation:**
   * Select relevant numerical and categorical clinical/socioeconomic predictors.
   * **Fairness Safeguard:** Exclude protected attributes (`sex`, `age`) from model input feature matrix `X` to eliminate direct demographic bias, while retaining them in split metadata for post-modeling fairness evaluation (demographic parity, equal opportunity).
2. **Stratified Splitting (70% Train, 15% Validation, 15% Test):** Partition data preserving target class ratios across all subsets with a fixed reproducible random seed.
3. **Leak-Proof Preprocessing:**
   * Fit imputers, scalers (`RobustScaler`), and one-hot encoders (`OneHotEncoder`) **strictly on training data (`X_train`)**.
   * Transform `X_val` and `X_test` using fitted rules without refitting.
4. **Artifact Serialization:**
   * Serialize fitted preprocessor to `preprocessing_pipeline.joblib`.
   * Export numerical scaling statistics to `numerical_scaling_stats.csv`.

### Inputs & Expected Outputs
* **Inputs:**
  * `data/interim/cleaned/ofphil_cleaned_tabular.csv`
* **Outputs:**
  * `data/processed/splits/train/X_train.csv`, `y_train.csv`
  * `data/processed/splits/validation/X_val.csv`, `y_val.csv`
  * `data/processed/splits/test/X_test.csv`, `y_test.csv`
  * `data/processed/features/preprocessing_pipeline.joblib`
  * `data/processed/preprocessing/numerical_scaling_stats.csv`

### 1. Environment Setup & Library Imports
Import scikit-learn preprocessing and model selection tools, joblib, and path resolvers.

In [1]:
from pathlib import Path
import json
import datetime
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'experiments').exists():
            return p
    return Path('../../..').resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())
print(f"Project root resolved to: {PROJECT_ROOT}")

Project root resolved to: /home/kristoper/Projects/Thesis/ofphil-classification-v2


### 2. Setup Paths & Load Cleaned Dataset
Configure processed data directory trees and load the cleaned dataset.

In [2]:
CLEANED_CSV = PROJECT_ROOT / 'data' / 'interim' / 'cleaned' / 'ofphil_cleaned_tabular.csv'
SPLITS_DIR = PROJECT_ROOT / 'data' / 'processed' / 'splits'
FEATURES_DIR = PROJECT_ROOT / 'data' / 'processed' / 'features'
AUDIT_CSV_PATH = PROJECT_ROOT / 'data' / 'processed' / 'preprocessing' / 'numerical_scaling_stats.csv'

for d in [SPLITS_DIR / 'train', SPLITS_DIR / 'validation', SPLITS_DIR / 'test', FEATURES_DIR, AUDIT_CSV_PATH.parent]:
    d.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CLEANED_CSV)
print(f"Loaded cleaned dataset with {len(df)} records and {df.shape[1]} columns.")

Loaded cleaned dataset with 1044 records and 62 columns.


### 3. Define Feature Subsets & Protected Attribute Safeguards
Separate numerical and categorical feature sets. Ensure protected attributes (`sex`, `age`) are excluded from model features `X`.

In [3]:
num_features = [
    'household_size',
    'number_of_dependents',
    'monthly_family_income',
    'monthly_household_obligations_expenses',
    'cost_of_treatment',
    'total_supporting_documents'
]

cat_features = [
    'employment_status',
    'residence_status',
    'educational_attainment',
    'diagnosis_severity',
    'type_of_treatment_surgery',
    'assistance_type'
]

protected_demographics = [c for c in ['age', 'sex', 'marital_status', 'sectoral_membership', 'disability_status'] if c in df.columns]

print(f"Selected Numerical Features ({len(num_features)}): {num_features}")
print(f"Selected Categorical Features ({len(cat_features)}): {cat_features}")
print(f"Protected Demographics (isolated from X): {protected_demographics}")

X = df[num_features + cat_features].copy()
y = (df['eligibility'] == 'ELIGIBLE').astype(int)

# Preserve demographic metadata aligned with records
demographics_df = df[protected_demographics].copy()

Selected Numerical Features (6): ['household_size', 'number_of_dependents', 'monthly_family_income', 'monthly_household_obligations_expenses', 'cost_of_treatment', 'total_supporting_documents']
Selected Categorical Features (6): ['employment_status', 'residence_status', 'educational_attainment', 'diagnosis_severity', 'type_of_treatment_surgery', 'assistance_type']
Protected Demographics (isolated from X): ['marital_status', 'sectoral_membership', 'disability_status']


### 4. Stratified Dataset Splitting (70% Train, 15% Validation, 15% Test)
Partition dataset using stratified random splitting with fixed seed `20261123`.

In [4]:
RANDOM_SEED = 20261123

# First split: 70% Train, 30% Temporary
X_train, X_temp, y_train, y_temp, demo_train, demo_temp = train_test_split(
    X, y, demographics_df, test_size=0.30, stratify=y, random_state=RANDOM_SEED
)

# Second split: 15% Validation, 15% Test (50% of temporary)
X_val, X_test, y_val, y_test, demo_val, demo_test = train_test_split(
    X_temp, y_temp, demo_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_SEED
)

print(f"Dataset Split Summary (Seed={RANDOM_SEED}):")
print(f" - Train set:      {len(X_train)} records ({len(X_train)/len(df)*100:.1f}%) | Positive Rate: {y_train.mean():.4f}")
print(f" - Validation set: {len(X_val)} records ({len(X_val)/len(df)*100:.1f}%) | Positive Rate: {y_val.mean():.4f}")
print(f" - Test set:       {len(X_test)} records ({len(X_test)/len(df)*100:.1f}%) | Positive Rate: {y_test.mean():.4f}")

Dataset Split Summary (Seed=20261123):
 - Train set:      730 records (69.9%) | Positive Rate: 0.5000
 - Validation set: 157 records (15.0%) | Positive Rate: 0.5032
 - Test set:       157 records (15.0%) | Positive Rate: 0.4968


### 5. Build Leak-Proof ColumnTransformer Preprocessor
Fit median imputer + `RobustScaler` on numerical features, and most frequent imputer + `OneHotEncoder` on categorical features **strictly on `X_train`**.

In [5]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_features),
        ('cat', cat_pipeline, cat_features)
    ],
    remainder='drop'
)

# Fit strictly on X_train to prevent data leakage
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

print(f"Transformed Feature Matrix Dimensions:")
print(f" - X_train_proc: {X_train_proc.shape}")
print(f" - X_val_proc:   {X_val_proc.shape}")
print(f" - X_test_proc:  {X_test_proc.shape}")

# Serialize the fitted pipeline
pipeline_path = FEATURES_DIR / 'preprocessing_pipeline.joblib'
joblib.dump(preprocessor, pipeline_path)
print(f"Fitted preprocessing pipeline serialized -> {pipeline_path}")

Transformed Feature Matrix Dimensions:
 - X_train_proc: (730, 31)
 - X_val_proc:   (157, 31)
 - X_test_proc:  (157, 31)
Fitted preprocessing pipeline serialized -> /home/kristoper/Projects/Thesis/ofphil-classification-v2/data/processed/features/preprocessing_pipeline.joblib


### 6. Export Dataset Splits & Numerical Scaling Statistics
Save CSV splits to disk and export scaling parameters to `data/processed/preprocessing/numerical_scaling_stats.csv`.

In [6]:
# Save raw split feature DataFrames (pre-imputation/pre-scaling for transparency) and target/demographics
X_train.to_csv(SPLITS_DIR / 'train' / 'X_train.csv', index=False)
y_train.to_csv(SPLITS_DIR / 'train' / 'y_train.csv', index=False)
demo_train.to_csv(SPLITS_DIR / 'train' / 'demographics_train.csv', index=False)

X_val.to_csv(SPLITS_DIR / 'validation' / 'X_val.csv', index=False)
y_val.to_csv(SPLITS_DIR / 'validation' / 'y_val.csv', index=False)
demo_val.to_csv(SPLITS_DIR / 'validation' / 'demographics_val.csv', index=False)

X_test.to_csv(SPLITS_DIR / 'test' / 'X_test.csv', index=False)
y_test.to_csv(SPLITS_DIR / 'test' / 'y_test.csv', index=False)
demo_test.to_csv(SPLITS_DIR / 'test' / 'demographics_test.csv', index=False)

# Also export leak-proof transformed feature matrices fitted strictly on X_train
encoded_cols = [str(c) for c in preprocessor.get_feature_names_out()]
pd.DataFrame(X_train_proc, columns=encoded_cols).to_csv(SPLITS_DIR / 'train' / 'X_train_processed.csv', index=False)
pd.DataFrame(X_val_proc, columns=encoded_cols).to_csv(SPLITS_DIR / 'validation' / 'X_val_processed.csv', index=False)
pd.DataFrame(X_test_proc, columns=encoded_cols).to_csv(SPLITS_DIR / 'test' / 'X_test_processed.csv', index=False)

print(f"Dataset splits (raw & processed) exported to: {SPLITS_DIR}")

# Export numerical scaling statistics strictly from training data
stats_rows = []
for col in num_features:
    s = pd.to_numeric(X_train[col], errors='coerce').dropna()
    q25 = float(s.quantile(0.25))
    q75 = float(s.quantile(0.75))
    stats_rows.append({
        'feature_name': col,
        'split_scope': 'train',
        'mean': round(float(s.mean()), 4),
        'std': round(float(s.std()), 4),
        'median': round(float(s.median()), 4),
        'iqr': round(q75 - q25, 4),
        'min': round(float(s.min()), 4),
        'max': round(float(s.max()), 4),
        'scaler_type': 'RobustScaler',
    })

stats_df = pd.DataFrame(stats_rows)
stats_df.to_csv(AUDIT_CSV_PATH, index=False)
print(f"Numerical scaling statistics exported -> {AUDIT_CSV_PATH}")
print(stats_df.to_string(index=False))

Dataset splits (raw & processed) exported to: /home/kristoper/Projects/Thesis/ofphil-classification-v2/data/processed/splits
Numerical scaling statistics exported -> /home/kristoper/Projects/Thesis/ofphil-classification-v2/data/processed/preprocessing/numerical_scaling_stats.csv
                          feature_name split_scope       mean        std  median      iqr    min     max  scaler_type
                        household_size       train     4.1481     1.7921     4.0     3.00    1.0     8.0 RobustScaler
                  number_of_dependents       train     1.5413     1.2984     1.0     2.00    0.0     5.0 RobustScaler
                 monthly_family_income       train 21801.7989 12907.8685 19355.0 17560.00    0.0 47500.0 RobustScaler
monthly_household_obligations_expenses       train 15118.8762  5989.8427 14570.0  8175.00 1200.0 39300.0 RobustScaler
                     cost_of_treatment       train 20839.2466 16844.3295 14200.0 16075.00 2650.0 68000.0 RobustScaler
            

### Findings & Observations
* **Leak-Proof Architecture:** Preprocessor fitted exclusively on `X_train`; validation and test sets strictly transformed.
* **Fairness Safeguard:** Protected attributes (`sex`, `age`) isolated from feature matrix `X`, but preserved in metadata for bias evaluation.
* **Stratified Conservation:** Exact 50% target class balance maintained across train, val, and test partitions.
* **Next Step:** Run `06_prepared_dataset_validation.ipynb` to execute invariant assertion checks on all prepared data assets.

### Completion Checklist
- [x] Feature subsets defined (6 numerical + 6 categorical).
- [x] Protected attributes isolated from feature matrix.
- [x] Stratified 70/15/15 train/val/test split executed with fixed seed.
- [x] `ColumnTransformer` fitted strictly on `X_train`.
- [x] `preprocessing_pipeline.joblib` serialized.
- [x] `numerical_scaling_stats.csv` exported to `data/processed/preprocessing/`.